<a href="https://colab.research.google.com/github/graemewales13/Langchain-Basics/blob/main/04-chat-memory.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aurelio-labs/langchain-course/blob/main/chapters/04-chat-memory.ipynb)

### LangChain Essentials

# Conversational Memory for OpenAI - LangChain

Conversational memory allows our chatbots and agents to remember previous interactions within a conversation. Without conversational memory, our chatbots would only ever be able to respond to the last message they received, essentially forgetting all previous messages with each new message.

Naturally, conversations require our chatbots to be able to respond over multiple interactions and refer to previous messages to understand the context of the conversation.

In [1]:
!pip install -U \
  "langchain==1.2.17" \
  "langchain-core==1.3.3" \
  "langchain-openai==1.2.1" \
  "langchain-community==0.4.1" \
  "langsmith>=0.3.45"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 113.1/113.1 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 543.9/543.9 kB 14.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.6/98.6 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 62.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 399.0/399.0 kB 26.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 45.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 173.8/173.8 kB 13.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 3.8 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
  Attempting uninstall: langsmith
    Found existing installation: langsmith 0.7.34
    Uninstalling langsmith-0.7.34:
      S

---

> ⚠️ We will be using OpenAI for this example allowing us to run everything via API. If you would like to use Ollama instead, check out the [Ollama LangChain Course](https://github.com/aurelio-labs/langchain-course/tree/main/notebooks/ollama).

---

---

> ⚠️ If using LangSmith, add your API key below:

In [2]:
import os
from getpass import getpass

os.environ["LANGCHAIN_API_KEY"] = os.getenv("LANGCHAIN_API_KEY") or \
    getpass("Enter LangSmith API Key: ")

os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_ENDPOINT"] = "https://api.smith.langchain.com"
os.environ["LANGCHAIN_PROJECT"] = "aurelioai-langchain-course-chat-memory-openai"

Enter LangSmith API Key: ··········


---

## LangChain's Memory Types

LangChain versions `0.0.x` consisted of various conversational memory types. Most of these are due for deprecation but still hold value in understanding the different approaches that we can take to building conversational memory.

Throughout the notebook we will be referring to these _older_ memory types and then rewriting them using the recommended `RunnableWithMessageHistory` class. We will learn about:

* `ConversationBufferMemory`: the simplest and most intuitive form of conversational memory, keeping track of a conversation without any additional bells and whistles.
* `ConversationBufferWindowMemory`: similar to `ConversationBufferMemory`, but only keeps track of the last `k` messages.
* `ConversationSummaryMemory`: rather than keeping track of the entire conversation, this memory type keeps track of a summary of the conversation.
* `ConversationSummaryBufferMemory`: merges the `ConversationSummaryMemory` and `ConversationTokenBufferMemory` types.

We'll work through each of these memory types in turn, and rewrite each one using the `RunnableWithMessageHistory` class.

## Initialize our LLM

Before jumping into our memory types, let's initialize our LLM. We will use OpenAI's `gpt-4o-mini` model, if you need an API key you can get one from [OpenAI's website](https://platform.openai.com/settings/organization/api-keys).

In [3]:
import os
from getpass import getpass
from langchain_openai import ChatOpenAI

os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY") or \
    getpass("Enter your OpenAI API key: ")

# For normal accurate responses
llm = ChatOpenAI(temperature=0.0, model="gpt-4o-mini")

Enter your OpenAI API key: ··········


## 1. `ConversationBufferMemory`

`ConversationBufferMemory` is the simplest form of conversational memory, it is _literally_ just a place that we store messages, and then use to feed messages into our LLM.

Let's start with LangChain's original `ConversationBufferMemory` object, we are setting `return_messages=True` to return the messages as a list of `ChatMessage` objects — unless using a non-chat model we would always set this to `True` as without it the messages are passed as a direct string which can lead to unexpected behavior from chat LLMs.

In [75]:
from langchain_core.chat_history import InMemoryChatMessageHistory

chat_map = {}

def get_chat_history(session_id: str) -> InMemoryChatMessageHistory:
    if session_id not in chat_map:
        chat_map[session_id] = InMemoryChatMessageHistory()
    return chat_map[session_id]

There are several ways that we can add messages to our memory, using the `save_context` method we can add a user query (via the `input` key) and the AI's response (via the `output` key). So, to create the following conversation:

```
User: Hi, my name is James
AI: Hey James, what's up? I'm an AI model called Zeta.
User: I'm researching the different types of conversational memory.
AI: That's interesting, what are some examples?
User: I've been looking at ConversationBufferMemory and ConversationBufferWindowMemory.
AI: That's interesting, what's the difference?
User: Buffer memory just stores the entire conversation, right?
AI: That makes sense, what about ConversationBufferWindowMemory?
User: Buffer window memory stores the last k messages, dropping the rest.
AI: Very cool!
```

We do:

In [76]:
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.runnables import ConfigurableFieldSpec

pipeline_with_history = RunnableWithMessageHistory(
    pipeline,
    get_session_history=get_chat_history,
    input_messages_key="query",
    history_messages_key="history",
    history_factory_config=[
        ConfigurableFieldSpec(
            id="session_id",
            annotation=str,
            name="Session ID",
            description="The session ID to use for the chat history",
            default="id_123",
        )
    ],
)

Before using the memory, we need to load in any variables for that memory type — in this case, there are none, so we just pass an empty dictionary:

In [77]:
for msg in [
    "I'm researching the different types of conversational memory.",
    "I have been looking at ConversationBufferMemory and ConversationBufferWindowMemory.",
    "Buffer memory just stores the entire conversation, right?",
    "Buffer window memory stores the last k messages, dropping the rest."
]:
    response = pipeline_with_history.invoke(
        {"query": msg},
        config={"configurable": {"session_id": "id_123"}},
    )
    print(getattr(response, "content", response))

print(chat_map["id_123"].messages)

Great! Conversational memory refers to the methods and techniques used by conversational agents (like chatbots and virtual assistants) to remember and utilize past interactions within a conversation. This helps create more coherent, context-aware, and personalized interactions. Here are some common types of conversational memory:

1. **Short-Term Memory (Session Memory):**
   - **Definition:** Holds information only for the duration of the current conversation session.
   - **Use case:** Remembering user inputs or conversational context temporarily to manage the flow within a single interaction.
   - **Example:** Remembering a user’s preference stated earlier in a single chat to provide relevant responses immediately afterward.

2. **Long-Term Memory:**
   - **Definition:** Stores information across multiple sessions to maintain context over time.
   - **Use case:** Retains user preferences, profile information, conversation history, or learning that helps in personalization and contin

In [80]:
agent.invoke(
    {"messages": [HumanMessage(content="what is my name again?")]}
)

{'messages': [HumanMessage(content='what is my name again?', additional_kwargs={}, response_metadata={}, id='bc975076-0e30-4bd2-9533-a65ab34653e3'),
  AIMessage(content='I’m not sure what your name is. Could you please tell me?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 15, 'prompt_tokens': 13, 'total_tokens': 28, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4.1-mini-2025-04-14', 'system_fingerprint': 'fp_aa0fd7085f', 'id': 'chatcmpl-DdFp0OOnyOb0MYZWWiiR46SsaKlWh', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019e07d6-da29-72e2-a234-bce70cb98210-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 13, 'output_tokens': 15, 'total_tokens': 28, 'input_token_details': {'audio'

With that, we've created our buffer memory. Before feeding it into our LLM let's quickly view the alternative method for adding messages to our memory. With this other method, we pass individual user and AI messages via the `add_user_message` and `add_ai_message` methods. To reproduce what we did above, we do:

In [78]:
from langchain.agents import create_agent
from langchain_core.messages import HumanMessage

agent = create_agent(
    model=llm,
    tools=[],
)

The outcome is exactly the same in either case. To pass this onto our LLM, we need to create a `ConversationChain` object — which is already deprecated in favor of the `RunnableWithMessageHistory` class, which we will cover in a moment.

In [23]:
agent.invoke({"input": "what is my name again?"})

{'messages': [HumanMessage(content='Hi, my name is James', additional_kwargs={}, response_metadata={}, id='394f40f5-44ea-42cc-92ba-1713789624f1'),
  AIMessage(content="Hey James, what's up? I'm an AI model called Zeta.", additional_kwargs={}, response_metadata={}, id='27878a38-8fb8-43a5-8b43-e0656a2f6728', tool_calls=[], invalid_tool_calls=[]),
  HumanMessage(content="I'm researching the different types of conversational memory.", additional_kwargs={}, response_metadata={}, id='0e79bbe6-1b7a-423b-a8e4-fe63c9c8e3d6'),
  AIMessage(content="That's interesting, what are some examples?", additional_kwargs={}, response_metadata={}, id='0357b580-d860-4498-b0b4-9103988d5c1b', tool_calls=[], invalid_tool_calls=[]),
  HumanMessage(content="I've been looking at ConversationBufferMemory and ConversationBufferWindowMemory.", additional_kwargs={}, response_metadata={}, id='25be0b49-2a8d-4b5c-a7c0-1a1e6de4ad6c'),
  AIMessage(content="That's interesting, what's the difference?", additional_kwargs={}, 

### `ConversationBufferMemory` with `RunnableWithMessageHistory`

As mentioned, the `ConversationBufferMemory` type is due for deprecation. Instead, we can use the `RunnableWithMessageHistory` class to implement the same functionality.

When implementing `RunnableWithMessageHistory` we will use **L**ang**C**hain **E**xpression **L**anguage (LCEL) and for this we need to define our prompt template and LLM components. Our `llm` has already been defined, so now we just define a `ChatPromptTemplate` object.

In [24]:
from langchain_core.prompts import (
    SystemMessagePromptTemplate,
    HumanMessagePromptTemplate,
    MessagesPlaceholder,
    ChatPromptTemplate,
)

system_prompt = "You are a helpful assistant called Zeta."

prompt_template = ChatPromptTemplate.from_messages([
    SystemMessagePromptTemplate.from_template(system_prompt),
    MessagesPlaceholder(variable_name="history"),
    HumanMessagePromptTemplate.from_template("{query}"),
])

We can link our `prompt_template` and our `llm` together to create a pipeline via LCEL.

In [25]:
pipeline = prompt_template | llm

Our `RunnableWithMessageHistory` requires our `pipeline` to be wrapped in a `RunnableWithMessageHistory` object. This object requires a few input parameters. One of those is `get_session_history`, which requires a function that returns a `ChatMessageHistory` object based on a session ID. We define this function ourselves:

In [26]:
from langchain_core.chat_history import InMemoryChatMessageHistory

chat_map = {}

def get_chat_history(session_id: str) -> InMemoryChatMessageHistory:
    if session_id not in chat_map:
        chat_map[session_id] = InMemoryChatMessageHistory()
    return chat_map[session_id]

We also need to tell our runnable which variable name to use for the chat history (ie `history`) and which to use for the user's query (ie `query`).

In [72]:
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.runnables import ConfigurableFieldSpec

pipeline_with_history = RunnableWithMessageHistory(
    pipeline,
    get_session_history=get_chat_history,
    input_messages_key="query",
    history_messages_key="history",
    history_factory_config=[
        ConfigurableFieldSpec(
            id="session_id",
            annotation=str,
            name="Session ID",
            description="The session ID to use for the chat history",
            default="id_123",
        )
    ],
)

Now we invoke our runnable:

In [73]:
response = pipeline_with_history.invoke(
    {"query": "Hi, my name is James"},
    config={"configurable": {"session_id": "id_123"}},
)

print(response.content)
chat_map["id_123"].messages

Hi James! It’s nice to meet you. What can I do for you today?


[HumanMessage(content='Hi, my name is James', additional_kwargs={}, response_metadata={}),
 AIMessage(content='Hello James! How can I assist you today?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 10, 'prompt_tokens': 26, 'total_tokens': 36, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4.1-mini-2025-04-14', 'system_fingerprint': 'fp_8c915479f7', 'id': 'chatcmpl-DdFeDTKVXe35xG2AnkS9y9mtRYBoC', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019e07cc-a51c-7d61-b37b-6a9dff34c1a8-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 26, 'output_tokens': 10, 'total_tokens': 36, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}

Our chat history will now be memorized and retrieved whenever we invoke our runnable with the same session ID.

In [74]:
response = pipeline_with_history.invoke(
    {"query": "What is my name again?"},
    config={"configurable": {"session_id": "id_123"}},
)

print(response.content)
chat_map["id_123"].messages

Your name is James. How can I assist you further?


[HumanMessage(content='Hi, my name is James', additional_kwargs={}, response_metadata={}),
 AIMessage(content='Hello James! How can I assist you today?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 10, 'prompt_tokens': 26, 'total_tokens': 36, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4.1-mini-2025-04-14', 'system_fingerprint': 'fp_8c915479f7', 'id': 'chatcmpl-DdFeDTKVXe35xG2AnkS9y9mtRYBoC', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019e07cc-a51c-7d61-b37b-6a9dff34c1a8-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 26, 'output_tokens': 10, 'total_tokens': 36, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}

We have now recreated the `ConversationBufferMemory` type using the `RunnableWithMessageHistory` class. Let's continue onto other memory types and see how these can be implemented.

## 2. `ConversationBufferWindowMemory`

The `ConversationBufferWindowMemory` type is similar to `ConversationBufferMemory`, but only keeps track of the last `k` messages. There are a few reasons why we would want to keep only the last `k` messages:

* More messages mean more tokens are sent with each request, more tokens increases latency _and_ cost.

* LLMs tend to perform worse when given more tokens, making them more likely to deviate from instructions, hallucinate, or _"forget"_ information provided to them. Conciseness is key to high performing LLMs.

* If we keep _all_ messages we will eventually hit the LLM's context window limit, by adding a window size `k` we can ensure we never hit this limit.

The buffer window solves many problems that we encounter with the standard buffer memory, while still being a very simple and intuitive form of conversational memory.

In [ ]:
from langchain.memory import ConversationBufferWindowMemory

memory = ConversationBufferWindowMemory(k=4, return_messages=True)

<ipython-input-16-c190440c16c0>:3: LangChainDeprecationWarning: Please see the migration guide at: https://python.langchain.com/docs/versions/migrating_memory/
  memory = ConversationBufferWindowMemory(k=4, return_messages=True)


We populate this memory using the same methods as before:

In [ ]:
memory.chat_memory.add_user_message("Hi, my name is James")
memory.chat_memory.add_ai_message("Hey James, what's up? I'm an AI model called Zeta.")
memory.chat_memory.add_user_message("I'm researching the different types of conversational memory.")
memory.chat_memory.add_ai_message("That's interesting, what are some examples?")
memory.chat_memory.add_user_message("I've been looking at ConversationBufferMemory and ConversationBufferWindowMemory.")
memory.chat_memory.add_ai_message("That's interesting, what's the difference?")
memory.chat_memory.add_user_message("Buffer memory just stores the entire conversation, right?")
memory.chat_memory.add_ai_message("That makes sense, what about ConversationBufferWindowMemory?")
memory.chat_memory.add_user_message("Buffer window memory stores the last k messages, dropping the rest.")
memory.chat_memory.add_ai_message("Very cool!")

memory.load_memory_variables({})

{'history': [HumanMessage(content="I'm researching the different types of conversational memory.", additional_kwargs={}, response_metadata={}),
  AIMessage(content="That's interesting, what are some examples?", additional_kwargs={}, response_metadata={}),
  HumanMessage(content="I've been looking at ConversationBufferMemory and ConversationBufferWindowMemory.", additional_kwargs={}, response_metadata={}),
  AIMessage(content="That's interesting, what's the difference?", additional_kwargs={}, response_metadata={}),
  HumanMessage(content='Buffer memory just stores the entire conversation, right?', additional_kwargs={}, response_metadata={}),
  AIMessage(content='That makes sense, what about ConversationBufferWindowMemory?', additional_kwargs={}, response_metadata={}),
  HumanMessage(content='Buffer window memory stores the last k messages, dropping the rest.', additional_kwargs={}, response_metadata={}),
  AIMessage(content='Very cool!', additional_kwargs={}, response_metadata={})]}

As before, we use the `ConversationChain` object (again, this is deprecated and we will rewrite it with `RunnableWithMessageHistory` in a moment).

In [ ]:
chain = ConversationChain(
    llm=llm,
    memory=memory,
    verbose=True
)

Now let's see if our LLM remembers our name:

In [ ]:
chain.invoke({"input": "what is my name again?"})



> Entering new ConversationChain chain...
Prompt after formatting:
The following is a friendly conversation between a human and an AI. The AI is talkative and provides lots of specific details from its context. If the AI does not know the answer to a question, it truthfully says it does not know.

Current conversation:
[HumanMessage(content="I'm researching the different types of conversational memory.", additional_kwargs={}, response_metadata={}), AIMessage(content="That's interesting, what are some examples?", additional_kwargs={}, response_metadata={}), HumanMessage(content="I've been looking at ConversationBufferMemory and ConversationBufferWindowMemory.", additional_kwargs={}, response_metadata={}), AIMessage(content="That's interesting, what's the difference?", additional_kwargs={}, response_metadata={}), HumanMessage(content='Buffer memory just stores the entire conversation, right?', additional_kwargs={}, response_metadata={}), AIMessage(content='That makes sense, what about 

{'input': 'what is my name again?',
 'history': [HumanMessage(content="I'm researching the different types of conversational memory.", additional_kwargs={}, response_metadata={}),
  AIMessage(content="That's interesting, what are some examples?", additional_kwargs={}, response_metadata={}),
  HumanMessage(content="I've been looking at ConversationBufferMemory and ConversationBufferWindowMemory.", additional_kwargs={}, response_metadata={}),
  AIMessage(content="That's interesting, what's the difference?", additional_kwargs={}, response_metadata={}),
  HumanMessage(content='Buffer memory just stores the entire conversation, right?', additional_kwargs={}, response_metadata={}),
  AIMessage(content='That makes sense, what about ConversationBufferWindowMemory?', additional_kwargs={}, response_metadata={}),
  HumanMessage(content='Buffer window memory stores the last k messages, dropping the rest.', additional_kwargs={}, response_metadata={}),
  AIMessage(content='Very cool!', additional_kw

The reason our LLM can no longer remember our name is because we have set the `k` parameter to `4`, meaning that only the last messages are stored in memory, as we can see above this does not include the first message where we introduced ourselves.

Based on the agent forgetting our name, we might wonder _why_ we would ever use this memory type compared to the standard buffer memory. Well, as with most things in AI, it is always a trade-off. Here we are able to support much longer conversations, use less tokens, and improve latency — but these come at the cost of forgetting non-recent messages.

### `ConversationBufferWindowMemory` with `RunnableWithMessageHistory`

To implement this memory type using the `RunnableWithMessageHistory` class, we can use the same approach as before. We define our `prompt_template` and `llm` as before, and then wrap our pipeline in a `RunnableWithMessageHistory` object.

For the window feature, we need to define a custom version of the `InMemoryChatMessageHistory` class that removes any messages beyond the last `k` messages.

In [31]:
from pydantic import BaseModel, Field
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.messages import BaseMessage

class BufferWindowMessageHistory(BaseChatMessageHistory, BaseModel):
    messages: list[BaseMessage] = Field(default_factory=list)
    k: int = Field(default_factory=int)

    def __init__(self, k: int):
        super().__init__(k=k)
        print(f"Initializing BufferWindowMessageHistory with k={k}")

    def add_messages(self, messages: list[BaseMessage]) -> None:
        """Add messages to the history, removing any messages beyond
        the last `k` messages.
        """
        self.messages.extend(messages)
        self.messages = self.messages[-self.k:]

    def clear(self) -> None:
        """Clear the history."""
        self.messages = []

In [32]:
chat_map = {}
def get_chat_history(session_id: str, k: int = 4) -> BufferWindowMessageHistory:
    print(f"get_chat_history called with session_id={session_id} and k={k}")
    if session_id not in chat_map:
        # if session ID doesn't exist, create a new chat history
        chat_map[session_id] = BufferWindowMessageHistory(k=k)
    # remove anything beyond the last
    return chat_map[session_id]

In [34]:
from langchain_core.messages import HumanMessage, AIMessage
from langchain_core.runnables import RunnableLambda

def invoke_with_history(inputs, config):
    session_id = config["configurable"]["session_id"]
    k = config["configurable"].get("k", 4)

    history = get_chat_history(session_id, k)

    result = pipeline.invoke(
        {
            "query": inputs["query"],
            "history": history.messages,
        }
    )

    response_text = getattr(result, "content", str(result))

    history.add_messages([
        HumanMessage(content=inputs["query"]),
        AIMessage(content=response_text),
    ])

    return result

pipeline_with_history = RunnableLambda(invoke_with_history)

Now we invoke our runnable, this time passing a `k` parameter via the `config` parameter.

In [35]:
pipeline_with_history.invoke(
    {"query": "Hi, my name is James"},
    config={"configurable": {"session_id": "id_k4", "k": 4}}
)

get_chat_history called with session_id=id_k4 and k=4
Initializing BufferWindowMessageHistory with k=4


AIMessage(content='Hello James! How can I assist you today?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 10, 'prompt_tokens': 26, 'total_tokens': 36, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4.1-mini-2025-04-14', 'system_fingerprint': 'fp_8c915479f7', 'id': 'chatcmpl-DdF7u7mqSeSqgB4RuI3oOC2sV8T3I', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019e07ae-10a9-7fa0-8fd8-68f182e13288-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 26, 'output_tokens': 10, 'total_tokens': 36, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

We can also modify the messages that are stored in memory by modifying the records inside the `chat_map` dictionary directly.

In [36]:
chat_map["id_k4"].clear()  # clear the history

# manually insert history
chat_map["id_k4"].add_user_message("Hi, my name is James")
chat_map["id_k4"].add_ai_message("I'm an AI model called Zeta.")
chat_map["id_k4"].add_user_message("I'm researching the different types of conversational memory.")
chat_map["id_k4"].add_ai_message("That's interesting, what are some examples?")
chat_map["id_k4"].add_user_message("I've been looking at ConversationBufferMemory and ConversationBufferWindowMemory.")
chat_map["id_k4"].add_ai_message("That's interesting, what's the difference?")
chat_map["id_k4"].add_user_message("Buffer memory just stores the entire conversation, right?")
chat_map["id_k4"].add_ai_message("That makes sense, what about ConversationBufferWindowMemory?")
chat_map["id_k4"].add_user_message("Buffer window memory stores the last k messages, dropping the rest.")
chat_map["id_k4"].add_ai_message("Very cool!")

chat_map["id_k4"].messages

[HumanMessage(content='Buffer memory just stores the entire conversation, right?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='That makes sense, what about ConversationBufferWindowMemory?', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='Buffer window memory stores the last k messages, dropping the rest.', additional_kwargs={}, response_metadata={}),
 AIMessage(content='Very cool!', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]

Now let's see at which `k` value our LLM remembers our name — from the above we can already see that with `k=4` our name is not mentioned, so when running with `k=4` we should expect the LLM to forget our name:

In [37]:
pipeline_with_history.invoke(
    {"query": "what is my name again?"},
    config={"configurable": {"session_id": "id_k4", "k": 4}}
)

get_chat_history called with session_id=id_k4 and k=4


AIMessage(content='I’m sorry, but I don’t have access to your name from our conversation so far. Could you please tell me your name?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 27, 'prompt_tokens': 79, 'total_tokens': 106, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4.1-mini-2025-04-14', 'system_fingerprint': 'fp_caf3f6485e', 'id': 'chatcmpl-DdF8ccFIFfTItA6y9loj1GMoPvoQX', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019e07ae-bc05-7693-bfab-b14b14b9cb21-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 79, 'output_tokens': 27, 'total_tokens': 106, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

Now let's initialize a new session with `k=14`.

In [38]:
pipeline_with_history.invoke(
    {"query": "Hi, my name is James"},
    config={"session_id": "id_k14", "k": 14}
)

get_chat_history called with session_id=id_k14 and k=14
Initializing BufferWindowMessageHistory with k=14


AIMessage(content='Hello James! How can I assist you today?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 10, 'prompt_tokens': 26, 'total_tokens': 36, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4.1-mini-2025-04-14', 'system_fingerprint': 'fp_8c915479f7', 'id': 'chatcmpl-DdF9COyYRiW2Gkoti12t4CdKQHgQr', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019e07af-4f80-7240-bffe-5c158879ffc1-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 26, 'output_tokens': 10, 'total_tokens': 36, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

We'll manually insert the remaining messages as before:

In [39]:
chat_map["id_k14"].add_user_message("I'm researching the different types of conversational memory.")
chat_map["id_k14"].add_ai_message("That's interesting, what are some examples?")
chat_map["id_k14"].add_user_message("I've been looking at ConversationBufferMemory and ConversationBufferWindowMemory.")
chat_map["id_k14"].add_ai_message("That's interesting, what's the difference?")
chat_map["id_k14"].add_user_message("Buffer memory just stores the entire conversation, right?")
chat_map["id_k14"].add_ai_message("That makes sense, what about ConversationBufferWindowMemory?")
chat_map["id_k14"].add_user_message("Buffer window memory stores the last k messages, dropping the rest.")
chat_map["id_k14"].add_ai_message("Very cool!")

chat_map["id_k14"].messages

[HumanMessage(content='Hi, my name is James', additional_kwargs={}, response_metadata={}),
 AIMessage(content='Hello James! How can I assist you today?', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content="I'm researching the different types of conversational memory.", additional_kwargs={}, response_metadata={}),
 AIMessage(content="That's interesting, what are some examples?", additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content="I've been looking at ConversationBufferMemory and ConversationBufferWindowMemory.", additional_kwargs={}, response_metadata={}),
 AIMessage(content="That's interesting, what's the difference?", additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='Buffer memory just stores the entire conversation, right?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='That makes sense, what about Conve

Now let's see if the LLM remembers our name:

In [40]:
pipeline_with_history.invoke(
    {"query": "what is my name again?"},
    config={"session_id": "id_k14", "k": 14}
)

get_chat_history called with session_id=id_k14 and k=14


AIMessage(content='Your name is James. How can I assist you further?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 12, 'prompt_tokens': 156, 'total_tokens': 168, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4.1-mini-2025-04-14', 'system_fingerprint': 'fp_05363a88ff', 'id': 'chatcmpl-DdF9ZV6zSHhQYfazgwRe3H7mlVk2z', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019e07af-ab71-7351-b27f-b47741d75e0e-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 156, 'output_tokens': 12, 'total_tokens': 168, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [41]:
chat_map["id_k14"].messages

[HumanMessage(content='Hi, my name is James', additional_kwargs={}, response_metadata={}),
 AIMessage(content='Hello James! How can I assist you today?', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content="I'm researching the different types of conversational memory.", additional_kwargs={}, response_metadata={}),
 AIMessage(content="That's interesting, what are some examples?", additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content="I've been looking at ConversationBufferMemory and ConversationBufferWindowMemory.", additional_kwargs={}, response_metadata={}),
 AIMessage(content="That's interesting, what's the difference?", additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='Buffer memory just stores the entire conversation, right?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='That makes sense, what about Conve

That's it! We've rewritten our buffer window memory using the recommended `RunnableWithMessageHistory` class.

## 3. `ConversationSummaryMemory`

Next up we have `ConversationSummaryMemory`, this memory type keeps track of a summary of the conversation rather than the entire conversation. This is useful for long conversations where we don't need to keep track of the entire conversation, but we do want to keep some thread of the full conversation.

As before, we'll start with the original memory class before reimplementing it with the `RunnableWithMessageHistory` class.

In [42]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableLambda

summary_map = {}

summary_prompt = ChatPromptTemplate.from_messages([
    ("system", "You maintain a concise running summary of a conversation."),
    ("human",
     "Current summary:\n{summary}\n\n"
     "New interaction:\nUser: {user_input}\nAssistant: {assistant_output}\n\n"
     "Return only the updated summary.")
])

def get_summary(session_id: str) -> str:
    return summary_map.get(session_id, "")

Unlike with the previous memory types, we need to provide an `llm` to initialize `ConversationSummaryMemory`. The reason for this is that we need an LLM to generate the conversation summaries.

Beyond this small tweak, using `ConversationSummaryMemory` is the same as with our previous memory types when using the deprecated `ConversationChain` object.

In [43]:
def invoke_with_summary(inputs, config):
    session_id = config.get("configurable", {}).get("session_id", "default")
    summary = get_summary(session_id)

    result = llm.invoke(
        [
            {"role": "system", "content": f"Conversation summary so far:\n{summary}"},
            {"role": "user", "content": inputs["input"]},
        ]
    )

    assistant_output = getattr(result, "content", str(result))

    updated_summary_msg = llm.invoke(
        summary_prompt.format_messages(
            summary=summary,
            user_input=inputs["input"],
            assistant_output=assistant_output,
        )
    )
    summary_map[session_id] = getattr(updated_summary_msg, "content", str(updated_summary_msg))

    return result

chain = RunnableLambda(invoke_with_summary)

Let's test:

In [44]:
chain.invoke(
    {"input": "hello there my name is James"},
    config={"configurable": {"session_id": "id_k4"}},
)
chain.invoke(
    {"input": "I am researching the different types of conversational memory."},
    config={"configurable": {"session_id": "id_k4"}},
)
chain.invoke(
    {"input": "I have been looking at ConversationBufferMemory and ConversationBufferWindowMemory."},
    config={"configurable": {"session_id": "id_k4"}},
)
chain.invoke(
    {"input": "Buffer memory just stores the entire conversation"},
    config={"configurable": {"session_id": "id_k4"}},
)
chain.invoke(
    {"input": "Buffer window memory stores the last k messages, dropping the rest."},
    config={"configurable": {"session_id": "id_k4"}},
)

AIMessage(content="Yes, exactly! ConversationBufferWindowMemory keeps only the most recent *k* messages (or turns) in memory and drops any earlier parts of the conversation. This helps limit the prompt size to a manageable context window, which is useful when you want to consider recent context but avoid long, growing histories.\n\nIf you'd like, I can:\n- Show you a code example of how to use ConversationBufferWindowMemory and set the window size,\n- Explain how to pick an appropriate window size based on your use case,\n- Compare its pros and cons in more detail, or\n- Help you with integrating it into a project.\n\nJust let me know!", additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 130, 'prompt_tokens': 231, 'total_tokens': 361, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_

We can see how the conversation summary varies with each new message. Let's see if the LLM is able to recall our name:

In [45]:
chain.invoke(
    {"input": "What is my name again?"},
    config={"configurable": {"session_id": "id_k4"}},
)

AIMessage(content='Your name is James. How can I assist you further, James?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 14, 'prompt_tokens': 288, 'total_tokens': 302, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4.1-mini-2025-04-14', 'system_fingerprint': 'fp_e31ab61991', 'id': 'chatcmpl-DdFHlGAmmXXU5jmuBCBPZhsf0cIyE', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019e07b7-69e1-79a0-933d-6b59977d1d65-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 288, 'output_tokens': 14, 'total_tokens': 302, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

As this information was stored in the summary the LLM successfully recalled our name. This may not always be the case, by summarizing the conversation we inevitably compress the full amount of information and so we may lose key details occasionally. Nonetheless, this is a great memory type for long conversations while retaining some key information.

### `ConversationSummaryMemory` with `RunnableWithMessageHistory`

Let's implement this memory type using the `RunnableWithMessageHistory` class. As with the window buffer memory, we need to define a custom implementation of the `InMemoryChatMessageHistory` class. We'll call this one `ConversationSummaryMessageHistory`.

In [46]:
from typing import Any
from pydantic import BaseModel, Field, ConfigDict

from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.messages import BaseMessage, SystemMessage
from langchain_core.prompts import ChatPromptTemplate


class ConversationSummaryMessageHistory(BaseChatMessageHistory, BaseModel):
    model_config = ConfigDict(arbitrary_types_allowed=True)

    messages: list[BaseMessage] = Field(default_factory=list)
    llm: Any

    def __init__(self, llm: Any):
        super().__init__(llm=llm)

    def add_messages(self, messages: list[BaseMessage]) -> None:
        """Summarize existing history plus new messages into a single SystemMessage."""
        existing_summary = ""
        if self.messages and isinstance(self.messages[0], SystemMessage):
            existing_summary = self.messages[0].content

        new_messages_text = "\n".join(
            f"{message.__class__.__name__}: {message.content}"
            for message in messages
            if hasattr(message, "content")
        )

        summary_prompt = ChatPromptTemplate.from_messages([
            (
                "system",
                "Given the existing conversation summary and the new messages, "
                "generate a new summary of the conversation. Ensure you maintain "
                "as much relevant information as possible."
            ),
            (
                "human",
                "Existing conversation summary:\n{existing_summary}\n\n"
                "New messages:\n{messages}"
            )
        ])

        new_summary = self.llm.invoke(
            summary_prompt.format_messages(
                existing_summary=existing_summary,
                messages=new_messages_text,
            )
        )

        self.messages = [SystemMessage(content=getattr(new_summary, "content", str(new_summary)))]

    def clear(self) -> None:
        """Clear the history."""
        self.messages = []

In [47]:
chat_map = {}

def get_chat_history(session_id: str, llm: Any) -> ConversationSummaryMessageHistory:
    if session_id not in chat_map:
        chat_map[session_id] = ConversationSummaryMessageHistory(llm=llm)
    return chat_map[session_id]

In [48]:
from langchain_core.messages import HumanMessage, AIMessage
from langchain_core.runnables import RunnableLambda

def invoke_with_history(inputs, config):
    cfg = config.get("configurable", {})
    session_id = cfg.get("session_id", "id_default")
    llm = cfg["llm"]

    history = get_chat_history(session_id, llm)

    result = pipeline.invoke(
        {
            "query": inputs["query"],
            "history": history.messages,
        }
    )

    response_text = getattr(result, "content", str(result))

    history.add_messages([
        HumanMessage(content=inputs["query"]),
        AIMessage(content=response_text),
    ])

    return result

pipeline_with_history = RunnableLambda(invoke_with_history)

Now we invoke our runnable, this time passing a `llm` parameter via the `config` parameter.

In [49]:
pipeline_with_history.invoke(
    {"query": "Hi, my name is James"},
    config={"configurable": {"session_id": "id_123", "llm": llm}},
)

AIMessage(content='Hello James! How can I assist you today?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 10, 'prompt_tokens': 26, 'total_tokens': 36, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4.1-mini-2025-04-14', 'system_fingerprint': 'fp_8c915479f7', 'id': 'chatcmpl-DdFMGvadUwsQKNFDIQmLw2t3IP5RU', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019e07bb-aac7-76f1-b250-ac06df44040e-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 26, 'output_tokens': 10, 'total_tokens': 36, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

Let's see what summary was generated:

In [50]:
chat_map["id_123"].messages

[SystemMessage(content='The user, James, introduced himself, and the AI greeted him and offered assistance.', additional_kwargs={}, response_metadata={})]

Let's continue the conversation and see if the summary is updated:

In [51]:
pipeline_with_history.invoke(
    {"query": "I'm researching the different types of conversational memory."},
    config={"configurable": {"session_id": "id_123", "llm": llm}},
)

chat_map["id_123"].messages

[SystemMessage(content='The user, James, introduced himself, and the AI greeted him and offered assistance. James then mentioned that he is researching different types of conversational memory. The AI explained various types of conversational memory including short-term memory, long-term memory, episodic memory, semantic memory, working memory, and external memory, and offered to provide more detailed explanations or examples upon request.', additional_kwargs={}, response_metadata={})]

So far so good! Let's continue with a few more messages before returning to the name question.

In [54]:
for msg in [
    "I have been looking at the deprecation of ConversationBufferMemory and ConversationBufferWindowMemory.",
    "i have added the new featues in my code",
]:
    pipeline_with_history.invoke(
        {"query": msg},
        config={"configurable": {"session_id": "id_123", "llm": llm}},
    )

Let's see the latest summary:

In [55]:
chat_map["id_123"].messages

[SystemMessage(content="The user, James, was initially exploring LangChain's conversational memory options, comparing `ConversationBufferMemory` and `ConversationBufferWindowMemory`. The AI clarified their differences and use cases. James then asked about their deprecation, and the AI confirmed that both have been deprecated in favor of more advanced memory systems, especially `ConversationSummaryMemory`, which summarizes conversation history to manage token limits effectively. The AI explained the reasons for deprecation, new approaches combining summaries with vector stores, and offered migration guidance.\n\nRecently, James mentioned that he has added the new features to his code. The AI responded positively, offering to review the implementation, provide feedback, or assist with any questions regarding the updated LangChain memory classes or conversational memory management.", additional_kwargs={}, response_metadata={})]

The information about our name has been maintained, so let's see if this is enough for our LLM to correctly recall our name.

In [56]:
pipeline_with_history.invoke(
    {"query": "What is my name again?"},
    config={"configurable": {"session_id": "id_123", "llm": llm}},
)

AIMessage(content='Your name is James. How can I assist you further today?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 13, 'prompt_tokens': 176, 'total_tokens': 189, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4.1-mini-2025-04-14', 'system_fingerprint': 'fp_286c30a665', 'id': 'chatcmpl-DdFPKxbuUmAU8h4C9oEU0yTnj9mHU', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019e07be-9353-7b50-8258-6dc34b053fe9-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 176, 'output_tokens': 13, 'total_tokens': 189, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

Perfect! We've successfully implemented the `ConversationSummaryMemory` type using the `RunnableWithMessageHistory` class.

## 4. `ConversationSummaryBufferMemory`

Our final memory type acts as a combination of `ConversationSummaryMemory` and `ConversationBufferMemory`. It keeps the buffer for the conversation up until the previous `n` tokens, anything beyond that limit is summarized then dropped from the buffer. Producing something like:


```
# ~~ a summary of previous interactions
The user named James introduced himself and the AI responded, introducing itself as an AI model called Zeta.
James then said he was researching the different types of conversational memory and Zeta asked for some
examples.
# ~~ the most recent messages
Human: I have been looking at ConversationBufferMemory and ConversationBufferWindowMemory.
AI: That's interesting, what's the difference?
Human: Buffer memory just stores the entire conversation
AI: That makes sense, what about ConversationBufferWindowMemory?
Human: Buffer window memory stores the last k messages, dropping the rest.
AI: Very cool!
```


In [57]:
from typing import Any
from pydantic import BaseModel, Field, ConfigDict

from langchain_core.messages import BaseMessage, SystemMessage, HumanMessage, AIMessage
from langchain_core.prompts import ChatPromptTemplate


class ConversationSummaryBufferMemory(BaseModel):
    model_config = ConfigDict(arbitrary_types_allowed=True)

    llm: Any
    max_token_limit: int = 300
    messages: list[BaseMessage] = Field(default_factory=list)
    summary: str = ""

    def _count_tokens(self, text: str) -> int:
        # simple approximation (you can swap for tiktoken if needed)
        return len(text.split())

    def load_memory(self):
        return {
            "history": [SystemMessage(content=self.summary)] + self.messages
        }

    def save_context(self, user_input: str, ai_output: str):
        # add to buffer
        self.messages.extend([
            HumanMessage(content=user_input),
            AIMessage(content=ai_output),
        ])

        # check token limit
        total_text = self.summary + " " + " ".join(
            m.content for m in self.messages if hasattr(m, "content")
        )

        if self._count_tokens(total_text) > self.max_token_limit:
            self._summarize()

    def _summarize(self):
        # summarize existing summary + buffer
        prompt = ChatPromptTemplate.from_messages([
            ("system", "Summarize the conversation, keeping important details."),
            ("human",
             "Current summary:\n{summary}\n\n"
             "New messages:\n{messages}")
        ])

        new_summary = self.llm.invoke(
            prompt.format_messages(
                summary=self.summary,
                messages="\n".join(m.content for m in self.messages),
            )
        )

        self.summary = getattr(new_summary, "content", str(new_summary))
        self.messages = []  # clear buffer after summarizing

As before, we set up the deprecated memory type using the `ConversationChain` object.

In [58]:
from langchain_core.runnables import RunnableLambda

memory = ConversationSummaryBufferMemory(
    llm=llm,
    max_token_limit=300,
)

def invoke_with_memory(inputs, config):
    mem = memory

    mem_vars = mem.load_memory()

    result = llm.invoke(
        mem_vars["history"] + [
            HumanMessage(content=inputs["input"])
        ]
    )

    output = getattr(result, "content", str(result))

    mem.save_context(inputs["input"], output)

    return result

chain = RunnableLambda(invoke_with_memory)

First invoke with a single message:

In [59]:
chain.invoke({"input": "Hi, my name is James"})

AIMessage(content='Hello James! How can I assist you today?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 10, 'prompt_tokens': 17, 'total_tokens': 27, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4.1-mini-2025-04-14', 'system_fingerprint': 'fp_aa0fd7085f', 'id': 'chatcmpl-DdFRHxHxPMvzfRi6mG3jz4DsdjSgC', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019e07c0-6aa9-7071-90d0-49ae369b72d1-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 17, 'output_tokens': 10, 'total_tokens': 27, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

Looks good so far, let's continue with a few more messages:

In [60]:
for msg in [
    "I'm researching the different types of conversational memory.",
    "I have been looking at ConversationBufferMemory and ConversationBufferWindowMemory.",
    "Buffer memory just stores the entire conversation",
    "Buffer window memory stores the last k messages, dropping the rest."
]:
    chain.invoke({"input": msg})

We can see with each new message the initial `SystemMessage` is updated with a new summary of the conversation. This initial `SystemMessage` is then followed by the most recent `AIMessage` and `HumanMessage` objects.

### `ConversationSummaryBufferMemory` with `RunnableWithMessageHistory`

As with the previous memory types, we will implement this memory type again using the `RunnableWithMessageHistory` class. In our implementation we will modify the buffer window to be based on the number of messages rather than number of tokens. This tweak will make our implementation more closely aligned with original buffer window.

We will implement all of this via a new `ConversationSummaryBufferMessageHistory` class.

In [61]:
from typing import Any
from pydantic import BaseModel, Field, ConfigDict

from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.messages import BaseMessage, SystemMessage
from langchain_core.prompts import ChatPromptTemplate


class ConversationSummaryBufferMessageHistory(BaseChatMessageHistory, BaseModel):
    model_config = ConfigDict(arbitrary_types_allowed=True)

    messages: list[BaseMessage] = Field(default_factory=list)
    llm: Any
    k: int = 4

    def __init__(self, llm: Any, k: int):
        super().__init__(llm=llm, k=k)

    def add_messages(self, messages: list[BaseMessage]) -> None:
        """Add messages, keep only the last `k`, and summarize anything dropped."""
        existing_summary: SystemMessage | None = None
        old_messages: list[BaseMessage] | None = None

        if self.messages and isinstance(self.messages[0], SystemMessage):
            existing_summary = self.messages.pop(0)

        self.messages.extend(messages)

        if len(self.messages) > self.k:
            old_messages = self.messages[:-self.k]
            self.messages = self.messages[-self.k:]

        if old_messages is None:
            return

        summary_prompt = ChatPromptTemplate.from_messages([
            (
                "system",
                "Given the existing conversation summary and the new messages, "
                "generate a new summary of the conversation. Ensure you maintain "
                "as much relevant information as possible."
            ),
            (
                "human",
                "Existing conversation summary:\n{existing_summary}\n\n"
                "New messages:\n{old_messages}"
            )
        ])

        new_summary = self.llm.invoke(
            summary_prompt.format_messages(
                existing_summary=(existing_summary.content if existing_summary else ""),
                old_messages="\n".join(
                    f"{m.__class__.__name__}: {m.content}"
                    for m in old_messages
                    if hasattr(m, "content")
                ),
            )
        )

        self.messages = [SystemMessage(content=new_summary.content)] + self.messages

    def clear(self) -> None:
        self.messages = []

Redefine the `get_chat_history` function to use our new `ConversationSummaryBufferMessageHistory` class.

In [62]:
chat_map = {}

def get_chat_history(session_id: str, llm: Any, k: int) -> ConversationSummaryBufferMessageHistory:
    if session_id not in chat_map:
        chat_map[session_id] = ConversationSummaryBufferMessageHistory(llm=llm, k=k)
    return chat_map[session_id]

Setup our pipeline with new configurable fields.

In [63]:
from langchain_core.messages import HumanMessage, AIMessage
from langchain_core.runnables import RunnableLambda
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.runnables import ConfigurableFieldSpec

pipeline_with_history = RunnableWithMessageHistory(
    pipeline,
    get_session_history=get_chat_history,
    input_messages_key="query",
    history_messages_key="history",
    history_factory_config=[
        ConfigurableFieldSpec(
            id="session_id",
            annotation=str,
            name="Session ID",
            description="The session ID to use for the chat history",
            default="id_default",
        ),
        ConfigurableFieldSpec(
            id="llm",
            annotation=Any,
            name="LLM",
            description="The LLM to use for the conversation summary",
            default=llm,
        ),
        ConfigurableFieldSpec(
            id="k",
            annotation=int,
            name="k",
            description="The number of messages to keep in the history",
            default=4,
        ),
    ],
)

Finally, we invoke our runnable:

In [64]:
pipeline_with_history.invoke(
    {"query": "Hi, my name is James"},
    config={"configurable": {"session_id": "id_123", "llm": llm, "k": 4}},
)

chat_map["id_123"].messages

[HumanMessage(content='Hi, my name is James', additional_kwargs={}, response_metadata={}),
 AIMessage(content='Hello James! How can I assist you today?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 10, 'prompt_tokens': 26, 'total_tokens': 36, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4.1-mini-2025-04-14', 'system_fingerprint': 'fp_8c915479f7', 'id': 'chatcmpl-DdFUt4hQgHil3M61QJB9rWZmGgfVK', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019e07c3-d50d-78b1-842a-caa3037ccf8a-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 26, 'output_tokens': 10, 'total_tokens': 36, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}

In [66]:
for i, msg in enumerate([
    "I'm researching the different types of conversational memory.",
    "I have been looking at ConversationBufferMemory and ConversationBufferWindowMemory.",
    "Buffer memory just stores the entire conversation",
    "Buffer window memory stores the last k messages, dropping the rest."
]):
    print(f"---\nMessage {i+1}\n---\n")
    pipeline_with_history.invoke(
        {"query": msg},
        config={"configurable": {"session_id": "id_123", "llm": llm, "k": 4}},
    )

---
Message 1
---

Great! Here's a brief overview of common types of conversational memory used in dialogue systems and language models:

1. **ConversationBufferMemory**  
   - Stores the entire conversation history from the start.  
   - Advantage: Full context is available for every exchange.  
   - Disadvantage: Can grow very large and lead to input size limits or slower processing.

2. **ConversationBufferWindowMemory**  
   - Stores only the last *k* messages (a sliding window).  
   - Advantage: Keeps context size manageable, focusing on recent relevant interactions.  
   - Disadvantage: Older context is discarded, which might lose important earlier information.

3. **ConversationSummaryMemory**  
   - Instead of storing raw messages, it maintains a running summary of past conversations.  
   - Advantage: Compact representation of long histories, reducing memory size.  
   - Disadvantage: Summaries may lose nuances or specific details.

4. **EntityMemory**  
   - Focuses on remem

There we go, we've successfully implemented the `ConversationSummaryBufferMemory` type using `RunnableWithMessageHistory`!